# Four-modality Mercury development experiment

Uses the verified seven-target MDR v4 release. Select one TARGET_ID per run;
all seven targets produce 252 records per model (36 each). Begin with Eminescu.
Conditions: NO_DATA, ALL_AVAILABLE (four modalities), RANDOM (five draws),
AGENT (one-shot ablation), AGENT_ITERATIVE (two cumulative acquisitions).

The historical three-modality notebook is unchanged. Literature search is paused
and is not part of this notebook. Do not combine these runs with old three-modality
results as a matched selector comparison. This is a development extension.

Preserved uploaded settings: Qwen3-VL-8B NF4, SDPA, 448-pixel views, 1024 answer
tokens, 512 selector tokens, answer repetition penalty 1.2, deterministic decoding.
Preflight detects input failure but cannot guarantee a full-length answer fits GPU RAM.
All source data remain read-only. Verify the release hash before loading the model.


## 1. Project and verified dataset


In [ ]:
import sys

assert sys.version_info >= (3, 12), sys.version

In [ ]:
import subprocess
from pathlib import Path

PROJECT = Path("/content/autonomous-modality-selection")
if (PROJECT / ".git").is_dir():
    dirty = subprocess.check_output(
        ["git", "-C", str(PROJECT), "status", "--porcelain"], text=True
    ).strip()
    if dirty:
        raise RuntimeError("The Colab checkout has uncommitted changes. Save them before updating.")
    subprocess.run(["git", "-C", str(PROJECT), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "merge", "--ff-only", "origin/main"], check=True)
elif PROJECT.exists():
    raise RuntimeError(f"Directory exists but is not a Git checkout; inspect it first: {PROJECT}")
else:
    # The repository is public; no token is needed for cloning.
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/liaojiqichi/autonomous-modality-selection.git",
            str(PROJECT),
        ],
        check=True,
    )
print(
    "Current commit:",
    subprocess.check_output(["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True).strip(),
)

In [ ]:
%cd /content/autonomous-modality-selection
%pip install -e ".[dev,pilot,multispectral]"

In [ ]:
!python -m ruff check .
!python -m ruff format --check .
!python -m pytest

In [ ]:
from autonomous_modality.colab_four_modality import (
    download_archive,
    extract_dataset,
)
from autonomous_modality.multispectral_release import verify_release

DATA_MODE = "download"  # Alternatives: "upload" or "local" (e.g. a mounted Drive ZIP).
ARCHIVE_PATH = Path("/content/mercury-four-modality-20261006.zip")
if DATA_MODE == "download":
    download_archive(ARCHIVE_PATH)
elif DATA_MODE == "upload":
    from google.colab import files

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one dataset ZIP.")
    filename = next(iter(uploaded))
    ARCHIVE_PATH = Path(filename).resolve()
    del uploaded  # Release the large uploaded byte buffer before loading the model.
elif DATA_MODE != "local":
    raise ValueError("Unknown DATA_MODE")
DATASET_ROOT = extract_dataset(ARCHIVE_PATH, Path("/content"))
release = verify_release(DATASET_ROOT)
print("Verified real development targets:", [(c.case_id, c.name) for c in release.cases])

In [ ]:
!nvidia-smi

## 2. Model environment
Skip installation if the model already works. Restart after installation if requested.

In [ ]:
%pip install -q "transformers>=4.57.1,<5" "accelerate>=1.2,<2" "bitsandbytes>=0.48,<1" pillow

In [ ]:
if "model" in globals():
    if (
        str(globals()["model"].config._name_or_path) != "Qwen/Qwen3-VL-8B-Instruct"
        or "processor" not in globals()
    ):
        raise RuntimeError("The loaded model differs from Qwen3-VL-8B; use a fresh runtime.")
    print("Reusing the loaded Qwen3-VL-8B without loading a second copy.")
else:
    import bitsandbytes
    import torch
    import transformers
    from transformers import (
        AutoProcessor,
        BitsAndBytesConfig,
        Qwen3VLForConditionalGeneration,
    )

    assert torch.cuda.is_available(), "Select a Colab GPU runtime first."

    print("GPU:", torch.cuda.get_device_name(0))
    print("PyTorch:", torch.__version__)
    print("Transformers:", transformers.__version__)
    print("bitsandbytes:", bitsandbytes.__version__)

    MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        dtype=torch.float16,
        device_map={"": 0},
        attn_implementation="sdpa",
        low_cpu_mem_usage=True,
    )
    model.eval()

    processor = AutoProcessor.from_pretrained(MODEL_ID)

    print(
        "GPU memory after model loading:",
        round(torch.cuda.memory_allocated() / 1024**3, 2),
        "GiB",
    )

## 3. Target, settings and compact four-modality inputs
Use a new RUN_LABEL; change TARGET_ID to process another released crater.


In [ ]:
import hashlib
import importlib.metadata
import json
import subprocess
import sys
from pathlib import Path, PureWindowsPath

import torch
from pydantic import BaseModel, ConfigDict

PROJECT = Path("/content/autonomous-modality-selection")
TARGET_ID = 16604  # Other IDs: 5031, 2179, 858, 4870, 19610, 1851.
CASE_DIR = DATASET_ROOT / f"crater-{TARGET_ID}"
RESULTS_ROOT = Path("/content/experiment-results")  # Use a Drive path if Drive is already mounted.
RUN_LABEL = (
    "four-modality-dev-001"  # Fresh English run; keep previous language conditions separate.
)
BUDGET = 4  # Primary budget 4; use 3 for sensitivity analysis.
SEED = 42
MAX_MODALITIES = 2
RANDOM_DRAWS = 5
IMAGE_EDGE = 448
ANSWER_TOKENS = 1024
SELECTOR_TOKENS = 512
MAX_INPUT_TOKENS = 8192
RECORD_VERSION = "colab-ap-four-modality-en-1"
ANSWER_REPETITION_PENALTY = 1.2
ITERATIVE_SELECTOR_TOKENS = 512
IS_DEVELOPMENT = True  # Eminescu has already been inspected and is development-only.


if not IS_DEVELOPMENT or BUDGET not in (3, 4):
    raise ValueError("This notebook is development-only; budget must be 3 or 4.")
sys.path.insert(0, str(PROJECT / "src"))
sys.path.insert(0, str(PROJECT / "extensions/evidence_views_v1/src"))

from autonomous_modality.development_inputs import (
    ANSWER_POLICY,
    ANSWER_POLICY_VERSION,
    INVENTORY_VERSION,
    development_inventory,
)
from autonomous_modality.experiments import scenario_seed
from autonomous_modality.models import InputDataModality
from autonomous_modality.pilot import CatalogueRow

MODEL_NAME = str(model.config._name_or_path)
ALLOWED_MODELS = ("Qwen/Qwen3-VL-8B-Instruct", "google/gemma-4-E4B-it")
if MODEL_NAME not in ALLOWED_MODELS:
    raise ValueError(f"Loaded model is outside this experiment: {MODEL_NAME}")
MODEL_TAG = "qwen3-vl-8b" if ALLOWED_MODELS[0] == MODEL_NAME else "gemma4-e4b"
OUTPUT = RESULTS_ROOT / RUN_LABEL / MODEL_TAG / f"crater-{TARGET_ID}" / f"budget-{BUDGET}"


def sha256(path: Path) -> str:
    """Hash files in chunks to limit memory use."""
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def json_digest(value: object) -> str:
    """Compare JSON configurations while rejecting NaN."""
    raw = json.dumps(
        value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False
    )
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


def write_new_json(path: Path, value: object) -> None:
    """Create a new file without overwriting existing results."""
    with path.open("x", encoding="utf-8") as stream:
        json.dump(value, stream, ensure_ascii=False, indent=2, allow_nan=False)


def portable_name(value: str) -> str:
    """Support Windows paths in records uploaded to Colab."""
    return PureWindowsPath(value).name


from autonomous_modality.colab_four_modality import prepare_compact_inputs
from autonomous_modality.multispectral_dataset import FourModalityManifest
from autonomous_modality.multispectral_release import verify_release

release = verify_release(DATASET_ROOT)
if TARGET_ID not in {entry.case_id for entry in release.cases}:
    raise ValueError("Target is not in the verified release.")
package = FourModalityManifest.model_validate_json((CASE_DIR / "manifest.json").read_bytes())
catalogue = CatalogueRow.model_validate_json((CASE_DIR / "core_views/catalogue.json").read_bytes())
if catalogue.id != TARGET_ID or package.case_id != TARGET_ID:
    raise ValueError("Target identity mismatch.")
CASE_ID, CASE_NAME = str(catalogue.id), catalogue.name
SOURCES = {
    "catalogue.json": CASE_DIR / "core_views/catalogue.json",
    "manifest.json": CASE_DIR / "manifest.json",
    "summary.json": CASE_DIR / "summary.json",
    "multispectral.npz": CASE_DIR / "multispectral.npz",
    "release.json": DATASET_ROOT / "release.json",
}
SOURCE_HASHES = {name: sha256(path) for name, path in SOURCES.items()}
question_path = PROJECT / "configs/mercury_questions_v1.json"
QUESTIONS = json.loads(question_path.read_text(encoding="utf-8"))["questions"]
assert [q["question_id"] for q in QUESTIONS] == ["Q1", "Q2", "Q3", "Q4"]


class ViewReceipt(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    version: str
    source_sha256: dict[str, str]
    implementation_sha256: dict[str, str]
    image_edge: int
    output_sha256: dict[str, str]


view_code = PROJECT / "extensions/evidence_views_v1/src/ams_evidence_views"
CODE_HASHES = {
    str(p.relative_to(PROJECT)): sha256(p)
    for root in (PROJECT / "src/autonomous_modality", view_code)
    for p in sorted(root.glob("*.py"))
}
VIEW_DIR = RESULTS_ROOT / RUN_LABEL / "shared_inputs" / CASE_ID
receipt = ViewReceipt.model_validate(prepare_compact_inputs(CASE_DIR, VIEW_DIR, IMAGE_EDGE))
receipt_path = VIEW_DIR / "receipt.json"
PREVIEWS = {
    key: VIEW_DIR / f"{key}.png"
    for key in (
        "optical_local",
        "optical_context",
        "elevation",
        "profile_east",
        "profile_north",
    )
}
TERRAIN_TEXT = (VIEW_DIR / "terrain.txt").read_text(encoding="utf-8")
MULTISPECTRAL_TEXT = (VIEW_DIR / "multispectral.txt").read_text(encoding="utf-8")
CATALOGUE_TEXT = json.dumps(catalogue.model_dump(), ensure_ascii=False)
OUTPUT.mkdir(parents=True, exist_ok=True)
ASSETS = {
    "CRATER_CATALOG": {
        "cost": 1,
        "description": (
            "Single historical target record; not a regional population or ground truth."
        ),
    },
    "OPTICAL_IMAGE": {
        "cost": 2,
        "description": (
            "Local and regional optical views; stretched DN, not composition or absolute age."
        ),
    },
    "TOPOGRAPHY": {
        "cost": 2,
        "description": (
            "Elevation map, fixed east/north profile plots, crop statistics "
            "and sampled numeric values in metres; not a measured crater depth. "
            "Raster itself is not directly analyzed by the LLM."
        ),
    },
}
ASSETS["MULTISPECTRAL_IMAGE"] = {
    "cost": 2,
    "description": (
        "Eight calibrated MDIS bands, 430-1000 nm: regional statistics. "
        "Geometric zones, not geological units. No mineral identification or executable raster."
    ),
}
for name, asset in ASSETS.items():
    inventory = (
        next(a.content_inventory for a in package.assets if a.modality.value == name)
        if name == "MULTISPECTRAL_IMAGE"
        else development_inventory(InputDataModality(name))
    )
    asset["content_inventory"] = inventory.model_dump(mode="json")
from autonomous_modality.development_inputs import SELECTOR_ALIGNMENT, SELECTOR_ALIGNMENT_VERSION

SELECTOR_TASK = (
    "Choose one supplied feasible input combination expected to support distinct, "
    "relevant analytical approaches and explanatory perspectives. " + SELECTOR_ALIGNMENT
)
# Shared, versioned answer policy imported above; all five conditions use it.
CONFIG = {
    "protocol": RECORD_VERSION,
    "split": "development",
    "case_id": CASE_ID,
    "case_name": CASE_NAME,
    "model": MODEL_NAME,
    "model_revision": getattr(model.config, "_commit_hash", None),
    "four_bit": bool(getattr(model, "is_loaded_in_4bit", False)),
    "seed": SEED,
    "budget": BUDGET,
    "maximum_modalities": MAX_MODALITIES,
    "random_draws": RANDOM_DRAWS,
    "image_edge": IMAGE_EDGE,
    "answer_tokens": ANSWER_TOKENS,
    "answer_repetition_penalty": ANSWER_REPETITION_PENALTY,
    "dataset_release_sha256": sha256(DATASET_ROOT / "release.json"),
    "development_extension": "four-modality-1.0",
    "selector_tokens": SELECTOR_TOKENS,
    "max_input_tokens": MAX_INPUT_TOKENS,
    "decoding": "greedy_one_beam",
    "thinking": "disabled",
    "language": "en",
    "language_protocol": "english-ap-1.0",
    "answer_policy_version": ANSWER_POLICY_VERSION,
    "inventory_version": INVENTORY_VERSION,
    "cost_unit": "legacy_ordinal_units",
    "iterative_selector_tokens": ITERATIVE_SELECTOR_TOKENS,
    "question_sha256": sha256(question_path),
    "source_sha256": SOURCE_HASHES,
    "view_receipt": receipt.model_dump(),
    "selector_task": SELECTOR_TASK,
    "answer_policy": ANSWER_POLICY,
    "assets": ASSETS,
    "repository_commit": subprocess.check_output(
        ["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True
    ).strip(),
    "implementation_sha256": CODE_HASHES,
}
print("Development case:", CASE_NAME, "| Model:", MODEL_NAME, "| Output:", OUTPUT)
print("Per model: 4 questions x (1 + 1 + 5 + 1 + 1) = 36 development records.")

## 4. Generation statistics, original selector and shared answer interface

In [ ]:
import copy
import time
from typing import Any, Literal

import torch
from pydantic import BaseModel, ConfigDict, Field


class GenerationTrace(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True, allow_inf_nan=False)
    messages: list[dict[str, Any]]
    max_new_tokens: int
    input_tokens: int | None = None
    output_tokens: int | None = None
    finish_reason: Literal["error", "eos", "length", "unknown"] = "error"
    elapsed_seconds: float = 0.0
    peak_gpu_gib: float | None = None
    text: str | None = None
    error: str | None = None


GENERATION_LOG = []


def generate_reply(
    messages: list[dict],
    max_new_tokens: int = 128,
    repetition_penalty: float = 1.0,
) -> str:
    """Keep the interface while logging outcomes, errors and resource usage for each call."""
    if max_new_tokens < 1:
        raise ValueError("max_new_tokens must be positive.")
    trace = GenerationTrace(messages=copy.deepcopy(messages), max_new_tokens=max_new_tokens)
    record = trace.model_dump()
    GENERATION_LOG.append(record)
    inputs = outputs = new_tokens = None
    started = time.perf_counter()
    try:
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        template_kwargs = {
            "tokenize": True,
            "add_generation_prompt": True,
            "return_dict": True,
            "return_tensors": "pt",
        }

        # The current Qwen3-VL-Instruct processor does not accept this argument.
        # Pass it only in the Gemma branch.
        if MODEL_NAME == "google/gemma-4-E4B-it":
            template_kwargs["enable_thinking"] = False

        inputs = processor.apply_chat_template(
            messages,
            **template_kwargs,
        ).to("cuda:0", dtype=model.dtype)
        n_input = int(inputs["input_ids"].shape[-1])
        trace.input_tokens = n_input
        if n_input > MAX_INPUT_TOKENS:
            raise ValueError(f"INPUT_TOKEN_LIMIT: {n_input} > {MAX_INPUT_TOKENS}")
        has_images = any(
            b.get("type") == "image" for message in messages for b in message["content"]
        )
        if has_images and not any("pixel" in name for name in inputs):
            raise ValueError("Processor output has no images; multimodal generation is blocked.")
        config = copy.deepcopy(model.generation_config)
        config.do_sample = False
        config.num_beams = 1
        config.temperature = None
        config.top_p = None
        config.top_k = None
        config.max_new_tokens = max_new_tokens
        config.repetition_penalty = repetition_penalty
        config.use_cache = True
        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                generation_config=config,
                use_model_defaults=False,
                do_sample=False,
                num_beams=1,
                max_new_tokens=max_new_tokens,
                use_cache=True,
                return_dict_in_generate=False,
            )
        new_tokens = outputs[:, n_input:]
        token_ids = new_tokens[0].tolist()
        trace.text = processor.batch_decode(new_tokens, skip_special_tokens=True)[0]
        trace.output_tokens = len(token_ids)
        eos = config.eos_token_id
        if eos is None:
            eos = processor.tokenizer.eos_token_id
        eos_ids = set(eos if isinstance(eos, (list, tuple)) else [eos])
        trace.finish_reason = (
            "eos"
            if token_ids and token_ids[-1] in eos_ids
            else "length"
            if len(token_ids) >= max_new_tokens
            else "unknown"
        )
        torch.cuda.synchronize()
        return trace.text
    except Exception as exc:
        trace.error = f"{type(exc).__name__}: {exc}"
        raise
    finally:
        trace.elapsed_seconds = time.perf_counter() - started
        trace.peak_gpu_gib = torch.cuda.max_memory_allocated() / 1024**3
        record.update(trace.model_dump())
        del inputs, outputs, new_tokens

In [ ]:
import random
from itertools import combinations

from pydantic import BaseModel, ConfigDict


class AgentChoice(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    option_id: str
    rationale: str = Field(min_length=1)


def valid_selection(selected: object) -> list[str]:
    """Validate RANDOM/AGENT selections; ALL_AVAILABLE has no selection-budget cap."""
    if not isinstance(selected, list) or not all(isinstance(x, str) for x in selected):
        raise ValueError("Selection must be a list of strings.")
    if not 1 <= len(selected) <= MAX_MODALITIES or len(set(selected)) != len(selected):
        raise ValueError("Invalid selection count or duplicate modalities.")
    if set(selected) - set(ASSETS):
        raise ValueError("Unknown modality.")
    if sum(ASSETS[x]["cost"] for x in selected) > BUDGET:
        raise ValueError("Selection exceeds the budget.")
    return sorted(selected)


FEASIBLE = [
    list(items)
    for n in range(1, MAX_MODALITIES + 1)
    for items in combinations(sorted(ASSETS), n)
    if sum(ASSETS[x]["cost"] for x in items) <= BUDGET
]
OPTIONS = {
    f"S{i + 1}": {"selected": items, "total_cost": sum(ASSETS[x]["cost"] for x in items)}
    for i, items in enumerate(FEASIBLE)
}
if not OPTIONS:
    raise ValueError("NO_FEASIBLE_OPTIONS")


def select_inputs(
    condition: str, question: str, question_id: str, draw_index: int = 0
) -> tuple[list[str], str]:
    """Retain three positional arguments and optional draw_index, without saved aliases."""
    if type(draw_index) is not int or not 0 <= draw_index < RANDOM_DRAWS:
        raise ValueError("Invalid draw_index.")
    if condition != "RANDOM" and draw_index != 0:
        raise ValueError("Non-random conditions have one attempt.")
    if condition == "NO_DATA":
        return [], "No additional scientific evidence."
    if condition == "ALL_AVAILABLE":
        return sorted(ASSETS), "All available modalities; non-budget-matched reference."
    if condition == "RANDOM":
        seed = scenario_seed(SEED, f"{CASE_ID}-{question_id}", BUDGET, draw_index)
        return valid_selection(random.Random(seed).choice(FEASIBLE)), (
            "Uniform feasible-subset draw with replacement."
        )
    if condition != "AGENT":
        raise ValueError(f"Unknown condition: {condition}")

    instruction = {
        "task": SELECTOR_TASK,
        "alignment_version": SELECTOR_ALIGNMENT_VERSION,
        "answer_requirements": next(
            q["answer_requirements"] for q in QUESTIONS if q["question_id"] == question_id
        ),
        "question": question,
        "modality_descriptions": ASSETS,
        "feasible_options": OPTIONS,
        "rules": [
            "Choose exactly one supplied option_id; do not change the option.",
            "Budget is a ceiling; spending less receives no separate reward.",
            "The catalog is one historical target record, not population data.",
            "Check content_inventory for actual and absent fields; unlisted fields are unknown.",
            "Inputs include plotted views, terrain samples and regional multispectral statistics, "
            "not executable rasters.",
            "Acknowledge representation limitations; "
            "focus on analytical approaches and explanations.",
            "Return only JSON with exactly option_id and a nonempty rationale.",
            "Write the rationale entirely in English.",
            "The rationale should be less than 200 characters.",
        ],
    }
    raw = generate_reply(
        [
            {
                "role": "user",
                "content": [{"type": "text", "text": json.dumps(instruction, ensure_ascii=False)}],
            }
        ],
        max_new_tokens=SELECTOR_TOKENS,
    )
    # GENERATION_LOG retains raw output for both successful and failed attempts.
    if GENERATION_LOG[-1]["finish_reason"] != "eos":
        raise ValueError("SELECTOR_OUTPUT_NOT_COMPLETE")
    parsed = AgentChoice.model_validate_json(raw)
    if parsed.option_id not in OPTIONS or not parsed.rationale.strip():
        raise ValueError("INVALID_AGENT_CHOICE")
    return valid_selection(OPTIONS[parsed.option_id]["selected"]), parsed.rationale


assert len(FEASIBLE) == (10 if BUDGET == 4 else 7)
assert select_inputs("NO_DATA", "test", "Q1") == ([], "No additional scientific evidence.")
print("Feasible combinations:", OPTIONS)
print(
    "Five random draws for Q1 (repeated combinations are valid):",
    [select_inputs("RANDOM", "test", "Q1", i)[0] for i in range(RANDOM_DRAWS)],
)

In [ ]:
def build_answer_messages(
    question: str, requirements: list[str], selected: list[str]
) -> list[dict]:
    """Route only selected evidence; exclude condition and selector metadata."""
    if len(set(selected)) != len(selected) or set(selected) - set(ASSETS):
        raise ValueError("Invalid evidence selection.")
    prompt = (
        ANSWER_POLICY
        + f"\nTarget: {CASE_NAME}\nQuestion:\n{question}\nRequirements:\n"
        + "\n".join(f"- {x}" for x in requirements)
    )
    images, texts = [], []
    if "CRATER_CATALOG" in selected:
        texts.append(
            "Historical catalogue record; morphology codes are not ground truth. "
            "Coordinates: degrees; diameter: km. "
            "int_shp: b=bowl, sh=slump hummocks, ff=flat floor; "
            "rim_shp: c=circular, sc=scalloped, t=terraced; "
            "cent_struc: cp=central peak, mp=multiple peaks, pi=central pit, "
            "pr=peak ring, mr=multi-ring, n=none; x=poorly imaged; "
            "rayed: y=yes, n=no. Unknown codes must not be guessed.\n" + CATALOGUE_TEXT
        )
    if "OPTICAL_IMAGE" in selected:
        for key in ("optical_local", "optical_context"):
            images.append({"type": "image", "image": str(PREVIEWS[key])})
            texts.append(f"Image {len(images)}: {key}; display-stretched optical view.")
    if "TOPOGRAPHY" in selected:
        for key in ("elevation", "profile_east", "profile_north"):
            images.append({"type": "image", "image": str(PREVIEWS[key])})
            texts.append(f"Image {len(images)}: {key}; elevation in metres, not crater depth.")
        texts.append(TERRAIN_TEXT)
    if "MULTISPECTRAL_IMAGE" in selected:
        texts.append(MULTISPECTRAL_TEXT)
    if not selected:
        texts.append("No additional scientific evidence is supplied.")
    # Both models receive the same content and order: images followed by text.
    content = [*images, {"type": "text", "text": prompt + "\n\n" + "\n\n".join(texts)}]
    return [{"role": "user", "content": content}]


def answer_question(question: str, requirements: list[str], selected: list[str]) -> str:
    """Call the loaded model with the token cap recorded in the run configuration."""
    return generate_reply(
        build_answer_messages(question, requirements, selected),
        max_new_tokens=ANSWER_TOKENS,
        repetition_penalty=ANSWER_REPETITION_PENALTY,
    )


# Check routing without calling the model.
for chosen, expected in [
    ([], 0),
    (["CRATER_CATALOG"], 0),
    (["OPTICAL_IMAGE"], 2),
    (["TOPOGRAPHY"], 3),
    (["MULTISPECTRAL_IMAGE"], 0),
    (sorted(ASSETS), 5),
]:
    content = build_answer_messages("fixture", [], chosen)[0]["content"]
    assert sum(x["type"] == "image" for x in content) == expected
print("Input routing verified: 0/0/2/3/5 images.")

## 5. Two-step agent and receipt-package compatibility layer
This cell defines interfaces and validates inputs only.

In [ ]:
from autonomous_modality.iterative import (
    ContentBlock,
    EvidenceObservation,
    IterativePolicy,
    IterativeSelection,
    run_iterative_selection,
)
from autonomous_modality.iterative_colab import notebook_generator
from autonomous_modality.models import (
    CraterQuestion,
    CraterReference,
    DataAssetProfile,
    InputDataModality,
    InputSelectionConstraints,
    InputSelectionRequest,
)

ITERATIVE_POLICY = IterativePolicy(
    cost_unit="legacy_ordinal_units",
    cost_definition=(
        "Ordinal costs: catalogue=1, optical=2, topography=2, multispectral=2; complete packages."
    ),
    selector_max_new_tokens=ITERATIVE_SELECTOR_TOKENS,
)
PINNED_RECEIPT_SHA256 = sha256(receipt_path)


def build_evidence_content(selected: list[str]) -> list[dict]:
    """Reuse the original routing, removing only the shared answer instruction prefix."""
    question = "Evidence routing placeholder"
    prefix = (
        ANSWER_POLICY + f"\nTarget: {CASE_NAME}\nQuestion:\n{question}\nRequirements:\n" + "\n\n"
    )
    content = copy.deepcopy(build_answer_messages(question, [], selected)[0]["content"])
    if content[-1]["type"] != "text" or not content[-1]["text"].startswith(prefix):
        raise ValueError("Answer routing changed; verify evidence-only extraction.")
    content[-1]["text"] = content[-1]["text"][len(prefix) :]
    return content


def load_notebook_evidence(modality: InputDataModality) -> EvidenceObservation:
    """Return actual receipt-verified input bytes through the original notebook routing."""
    if sha256(receipt_path) != PINNED_RECEIPT_SHA256:
        raise ValueError("Input receipt changed during acquisition.")
    for name, digest in receipt.output_sha256.items():
        if sha256(VIEW_DIR / name) != digest:
            raise ValueError(f"Input view changed: {name}")
    if sha256(SOURCES["catalogue.json"]) != SOURCE_HASHES["catalogue.json"]:
        raise ValueError("Catalogue changed during acquisition.")
    actual_catalogue = CatalogueRow.model_validate_json(
        SOURCES["catalogue.json"].read_text(encoding="utf-8")
    )
    if json.dumps(actual_catalogue.model_dump(), ensure_ascii=False) != CATALOGUE_TEXT:
        raise ValueError("In-memory catalogue differs from pinned input.")
    if (VIEW_DIR / "terrain.txt").read_text(encoding="utf-8") != TERRAIN_TEXT:
        raise ValueError("In-memory terrain differs from pinned input.")
    if (VIEW_DIR / "multispectral.txt").read_text(encoding="utf-8") != MULTISPECTRAL_TEXT:
        raise ValueError("In-memory multispectral evidence changed.")
    files_by_modality = {
        "MULTISPECTRAL_IMAGE": {"multispectral.txt": receipt.output_sha256["multispectral.txt"]},
        "CRATER_CATALOG": {"catalogue.json": SOURCE_HASHES["catalogue.json"]},
        "OPTICAL_IMAGE": {
            name: receipt.output_sha256[name]
            for name in ("optical_local.png", "optical_context.png")
        },
        "TOPOGRAPHY": {
            name: receipt.output_sha256[name]
            for name in ("elevation.png", "profile_east.png", "profile_north.png", "terrain.txt")
        },
    }
    return EvidenceObservation(
        modality=modality,
        blocks=[ContentBlock.model_validate(b) for b in build_evidence_content([modality.value])],
        evidence_sha256=files_by_modality[modality.value],
        package_sha256=PINNED_RECEIPT_SHA256,
    )


def select_inputs_iterative(question: str, question_id: str) -> IterativeSelection:
    """A separate entry point; the original select_inputs function remains unchanged."""
    spec = next(q for q in QUESTIONS if q["question_id"] == question_id)
    request = InputSelectionRequest(
        question=CraterQuestion(
            question_id=f"{CASE_ID}-{question_id}",
            text=question,
            question_type=spec["question_type"],
            answer_requirements=spec["answer_requirements"],
            crater=CraterReference(
                crater_id=CASE_ID,
                name=CASE_NAME,
                latitude=catalogue.lat_n,
                longitude=catalogue.lon_e_0,
            ),
        ),
        assets=[
            DataAssetProfile(
                asset_id=name,
                modality=name,
                title=name,
                estimated_cost=asset["cost"],
                limitations=[asset["description"]],
                content_inventory=asset["content_inventory"],
            )
            for name, asset in sorted(ASSETS.items())
        ],
        constraints=InputSelectionConstraints(
            maximum_modalities=MAX_MODALITIES,
            maximum_total_cost=BUDGET,
        ),
    )
    return run_iterative_selection(
        request,
        ITERATIVE_POLICY,
        notebook_generator(generate_reply, GENERATION_LOG),
        load_notebook_evidence,
        model_id=MODEL_NAME,
        execution_kind="model",
    )


# Routing checks only: no model call. The final answer still uses answer_question
# for ALL five conditions, preserving the exact input ordering and answer prompt.
for modality, images in [
    ("CRATER_CATALOG", 0),
    ("OPTICAL_IMAGE", 2),
    ("TOPOGRAPHY", 3),
    ("MULTISPECTRAL_IMAGE", 0),
]:
    observation = load_notebook_evidence(InputDataModality(modality))
    assert sum(b.type == "image" for b in observation.blocks) == images
    assert all(ANSWER_POLICY not in (b.text or "") for b in observation.blocks)
print("Two-step interface ready; real input routing verified without model inference.")

## 6. Run the five-condition experiment
This cell starts GPU inference. The original AGENT remains an ablation.

In [ ]:
import gc
import inspect
import traceback
from typing import Literal

from pydantic import BaseModel, ConfigDict, Field, model_validator

from autonomous_modality.protocol import ExperimentProtocol

CONDITIONS = ["NO_DATA", "ALL_AVAILABLE", "RANDOM", "AGENT", "AGENT_ITERATIVE"]


class Attempt(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True, allow_inf_nan=False)
    record_version: Literal["colab-ap-four-modality-en-1"] = "colab-ap-four-modality-en-1"
    run_sha256: str
    model: str
    case_id: str
    question_id: str
    question: str
    condition: Literal["NO_DATA", "ALL_AVAILABLE", "RANDOM", "AGENT", "AGENT_ITERATIVE"]
    draw_index: int = Field(ge=0, le=4)
    selection_seed: int | None = None
    status: Literal["success", "truncated", "incomplete", "error"]
    selected: list[str]
    selection_rationale: str
    iterative_selection: IterativeSelection | None = None
    input_cost: int = Field(ge=0)
    elapsed_seconds: float = Field(ge=0)
    selection_seconds: float = Field(ge=0)
    answer_seconds: float = Field(ge=0)
    peak_gpu_gib: float | None = None
    selection_generations: list[GenerationTrace]
    answer_generations: list[GenerationTrace]
    answer: str | None = None
    finish_reason: Literal["eos", "length", "unknown"] | None = None
    error_stage: Literal["selection", "answer"] | None = None
    error_type: str | None = None
    error: str | None = None
    traceback: str | None = None

    @model_validator(mode="after")
    def check_record(self) -> "Attempt":
        if self.condition != "RANDOM" and self.draw_index != 0:
            raise ValueError("Non-random conditions cannot contain extra draws.")
        if self.condition == "NO_DATA" and self.selected:
            raise ValueError("NO_DATA must contain no additional evidence.")
        if set(self.selected) - set(ASSETS) or len(set(self.selected)) != len(self.selected):
            raise ValueError("Record contains invalid modalities.")
        if self.input_cost != sum(ASSETS[x]["cost"] for x in self.selected):
            raise ValueError("Cost mismatch.")
        if self.selected and self.condition in {"RANDOM", "AGENT", "AGENT_ITERATIVE"}:
            valid_selection(self.selected)
        if self.status != "error":
            if self.condition == "ALL_AVAILABLE" and self.selected != sorted(ASSETS):
                raise ValueError("ALL_AVAILABLE is missing modalities.")
            if self.condition not in {"NO_DATA", "AGENT_ITERATIVE"} and not self.selected:
                raise ValueError("A data condition is missing its selection.")
            if not self.answer or not self.answer_generations:
                raise ValueError("Answer text or generation record is missing.")
            if self.status == "success" and self.finish_reason != "eos":
                raise ValueError("An answer without EOS cannot be marked successful.")
        if self.condition == "AGENT_ITERATIVE":
            if self.status != "error" and self.iterative_selection is None:
                raise ValueError("Missing iterative acquisition trace.")
            if self.iterative_selection is not None:
                trace = self.iterative_selection
                if sorted(m.value for m in trace.accessed_modalities) != self.selected:
                    raise ValueError("Iterative access history differs from selected evidence.")
                if trace.cumulative_cost != self.input_cost:
                    raise ValueError("Iterative cumulative cost mismatch.")
                if self.status != "error" and trace.status != "ready":
                    raise ValueError("Cannot answer after failed acquisition.")
        elif self.iterative_selection is not None:
            raise ValueError("Iterative trace on another condition.")
        return self


protocol_path = PROJECT / "configs/experiment_protocol.json"
protocol = ExperimentProtocol.model_validate_json(protocol_path.read_text(encoding="utf-8"))
live_settings = {
    "model": str(model.config._name_or_path),
    "seed": SEED,
    "budget": BUDGET,
    "maximum_modalities": MAX_MODALITIES,
    "random_draws": RANDOM_DRAWS,
    "image_edge": IMAGE_EDGE,
    "answer_tokens": ANSWER_TOKENS,
    "answer_repetition_penalty": ANSWER_REPETITION_PENALTY,
    "selector_tokens": SELECTOR_TOKENS,
    "max_input_tokens": MAX_INPUT_TOKENS,
    "iterative_selector_tokens": ITERATIVE_SELECTOR_TOKENS,
}
if any(CONFIG[key] != value for key, value in live_settings.items()):
    raise ValueError("Settings changed; rerun from section 3 in order.")
if MODEL_NAME not in protocol.models:
    raise ValueError("Model differs from the project protocol.")
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Require inspect.getsource for code fingerprints; fail explicitly if source is unavailable.
tracked_objects = (
    sha256,
    json_digest,
    write_new_json,
    portable_name,
    generate_reply,
    valid_selection,
    select_inputs,
    build_answer_messages,
    answer_question,
    Attempt.check_record,
    build_evidence_content,
    load_notebook_evidence,
    select_inputs_iterative,
)
try:
    function_sources = {x.__name__: inspect.getsource(x) for x in tracked_objects}
except (OSError, TypeError) as exc:
    raise RuntimeError("Cell source unavailable; rerun the notebook cells in order.") from exc

metadata = {
    "config": CONFIG,
    "protocol": protocol.model_dump(mode="json"),
    "protocol_sha256": sha256(protocol_path),
    "functions": function_sources,
    "iterative_policy": ITERATIVE_POLICY.model_dump(mode="json"),
    "condition_roles": {"AGENT": "one_shot_ablation", "AGENT_ITERATIVE": "primary_agent"},
    "schemas": {
        cls.__name__: cls.model_json_schema()
        for cls in (GenerationTrace, AgentChoice, Attempt, ViewReceipt, IterativeSelection)
    },
    "options": OPTIONS,
    "model_config": json.loads(model.config.to_json_string()),
    "generation_defaults": json.loads(model.generation_config.to_json_string()),
    "processor_class": type(processor).__name__,
    "image_processor": json.loads(processor.image_processor.to_json_string()),
    "chat_template": getattr(processor, "chat_template", None),
    "tokenizer_chat_template": getattr(processor.tokenizer, "chat_template", None),
    "tokenizer_class": type(processor.tokenizer).__name__,
    "environment": {
        name: importlib.metadata.version(name)
        for name in (
            "torch",
            "transformers",
            "bitsandbytes",
            "accelerate",
            "numpy",
            "Pillow",
            "rasterio",
            "matplotlib",
            "pydantic",
        )
    },
    "gpu": torch.cuda.get_device_name(0),
    "cuda": torch.version.cuda,
    "deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),
}
# Recheck actual inputs; changed inputs require a new RUN_LABEL.
if {name: sha256(path) for name, path in SOURCES.items()} != SOURCE_HASHES:
    raise ValueError("Source files changed.")
for name, digest in receipt.output_sha256.items():
    if sha256(VIEW_DIR / name) != digest:
        raise ValueError(f"Input view changed: {name}")
# Within a RUN_LABEL, both models share questions, input bytes, prompts and random choices.
# Across sessions, use persistent RESULTS_ROOT or restore the exported experiment directory.
paired_design = {
    "case_id": CASE_ID,
    "source_sha256": SOURCE_HASHES,
    "view_sha256": receipt.output_sha256,
    "question_sha256": CONFIG["question_sha256"],
    "budget": BUDGET,
    "maximum_modalities": MAX_MODALITIES,
    "answer_tokens": ANSWER_TOKENS,
    "answer_repetition_penalty": ANSWER_REPETITION_PENALTY,
    "selector_tokens": SELECTOR_TOKENS,
    "answer_policy": ANSWER_POLICY,
    "selector_task": SELECTOR_TASK,
    "options": OPTIONS,
    "iterative_policy": ITERATIVE_POLICY.model_dump(mode="json"),
    "conditions": CONDITIONS,
    "random_choices": {
        q["question_id"]: [
            select_inputs("RANDOM", q["text"], q["question_id"], i)[0] for i in range(RANDOM_DRAWS)
        ]
        for q in QUESTIONS
    },
}
paired_path = RESULTS_ROOT / RUN_LABEL / f"paired-{CASE_ID}-b{BUDGET}.json"
if paired_path.exists():
    if json.loads(paired_path.read_text(encoding="utf-8")) != paired_design:
        raise ValueError("Shared design or inputs differ between models; comparison is blocked.")
else:
    write_new_json(paired_path, paired_design)
metadata["paired_design_sha256"] = sha256(paired_path)
metadata_path = OUTPUT / "run_metadata.json"
if metadata_path.exists():
    if json.loads(metadata_path.read_text(encoding="utf-8")) != metadata:
        raise ValueError("Code, model or configuration changed; use a new RUN_LABEL.")
else:
    if list(OUTPUT.glob("Q*__*.json")):
        raise ValueError("Existing results have no matching configuration; use a new RUN_LABEL.")
    write_new_json(metadata_path, metadata)
RUN_SHA256 = sha256(metadata_path)

# Test the largest input briefly; a full 1024-token answer may still exceed GPU memory.
# Stop on failure; keep evidence and selection policy fixed across conditions.
preflight_path = OUTPUT / "preflight.json"
if not preflight_path.exists():
    GENERATION_LOG.clear()
    try:
        generate_reply(
            build_answer_messages(
                "Suggest research using these inputs. Answer in English without measurements.",
                [],
                sorted(ASSETS),
            ),
            max_new_tokens=32,
        )
        preflight = {
            "run_sha256": RUN_SHA256,
            "status": "completed_input_smoke_test",
            "generation": GENERATION_LOG[-1],
        }
    except Exception:
        write_new_json(
            preflight_path,
            {
                "run_sha256": RUN_SHA256,
                "status": "error",
                "generation": GENERATION_LOG[-1] if GENERATION_LOG else None,
            },
        )
        raise
    write_new_json(preflight_path, preflight)
else:
    preflight = json.loads(preflight_path.read_text(encoding="utf-8"))
    if preflight.get("run_sha256") != RUN_SHA256 or preflight.get("status") == "error":
        raise ValueError("Preflight changed or failed; resolve the issue and use a new RUN_LABEL.")

for spec in QUESTIONS:
    question_id = spec["question_id"]
    question = spec["text"].replace("{crater}", CASE_NAME)
    for condition in CONDITIONS:
        for draw_index in range(RANDOM_DRAWS if condition == "RANDOM" else 1):
            destination = OUTPUT / f"{question_id}__{condition}__r{draw_index + 1:02d}.json"
            selection_seed = (
                scenario_seed(SEED, f"{CASE_ID}-{question_id}", BUDGET, draw_index)
                if condition == "RANDOM"
                else None
            )
            if destination.exists():
                old = Attempt.model_validate_json(destination.read_text(encoding="utf-8"))
                actual = (
                    old.run_sha256,
                    old.model,
                    old.case_id,
                    old.question_id,
                    old.condition,
                    old.draw_index,
                    old.selection_seed,
                )
                expected = (
                    RUN_SHA256,
                    MODEL_NAME,
                    CASE_ID,
                    question_id,
                    condition,
                    draw_index,
                    selection_seed,
                )
                if actual != expected:
                    raise ValueError(f"Existing result identity mismatch: {destination.name}")
                print(
                    "Keeping existing attempt (including failures):", destination.name, old.status
                )
                continue

            GENERATION_LOG.clear()
            started = stage_started = time.perf_counter()
            stage = "selection"
            selected, rationale = [], ""
            iterative_selection = None
            selection_seconds = answer_seconds = 0.0
            selection_generations, answer_generations = [], []
            outcome = {}
            try:
                if condition == "AGENT_ITERATIVE":
                    iterative_selection = select_inputs_iterative(question, question_id)
                    selected = sorted(m.value for m in iterative_selection.accessed_modalities)
                    rationale = " | ".join(
                        t.action.reason for t in iterative_selection.turns if t.action is not None
                    )
                    if iterative_selection.status != "ready":
                        raise ValueError(
                            f"{iterative_selection.stop_reason}: {iterative_selection.error}"
                        )
                else:
                    selected, rationale = select_inputs(
                        condition, question, question_id, draw_index=draw_index
                    )
                selection_seconds = time.perf_counter() - stage_started
                selection_generations = copy.deepcopy(GENERATION_LOG)
                GENERATION_LOG.clear()
                stage, stage_started = "answer", time.perf_counter()
                if iterative_selection is not None:
                    for turn in iterative_selection.turns:
                        if turn.observation is not None:
                            current = load_notebook_evidence(turn.observation.modality)
                            if current != turn.observation:
                                raise ValueError("Acquired evidence changed before answering.")
                answer = answer_question(question, spec["answer_requirements"], selected)
                answer_seconds = time.perf_counter() - stage_started
                answer_generations = copy.deepcopy(GENERATION_LOG)
                if not answer_generations or not answer.strip():
                    raise ValueError("EMPTY_OR_UNLOGGED_ANSWER")
                finish = answer_generations[-1]["finish_reason"]
                outcome = {
                    "status": {"eos": "success", "length": "truncated"}.get(finish, "incomplete"),
                    "answer": answer,
                    "finish_reason": finish,
                }
            except Exception as exc:
                if stage == "selection":
                    selection_seconds = time.perf_counter() - stage_started
                    selection_generations = copy.deepcopy(GENERATION_LOG)
                else:
                    answer_seconds = time.perf_counter() - stage_started
                    answer_generations = copy.deepcopy(GENERATION_LOG)
                outcome = {
                    "status": "error",
                    "error_stage": stage,
                    "error_type": type(exc).__name__,
                    "error": str(exc),
                    "traceback": traceback.format_exc(),
                }
            peaks = [
                g["peak_gpu_gib"]
                for g in selection_generations + answer_generations
                if g.get("peak_gpu_gib") is not None
            ]
            result = Attempt(
                run_sha256=RUN_SHA256,
                model=MODEL_NAME,
                case_id=CASE_ID,
                question_id=question_id,
                question=question,
                condition=condition,
                draw_index=draw_index,
                selection_seed=selection_seed,
                selected=selected,
                selection_rationale=rationale,
                iterative_selection=iterative_selection,
                input_cost=sum(ASSETS[x]["cost"] for x in selected),
                elapsed_seconds=time.perf_counter() - started,
                selection_seconds=selection_seconds,
                answer_seconds=answer_seconds,
                peak_gpu_gib=max(peaks, default=None),
                selection_generations=selection_generations,
                answer_generations=answer_generations,
                **outcome,
            )
            temporary = destination.with_suffix(".json.tmp")
            write_new_json(temporary, result.model_dump(mode="json"))
            if destination.exists():
                raise RuntimeError("Do not run concurrent notebooks writing to the same RUN_LABEL.")
            temporary.replace(destination)
            print(
                question_id,
                condition,
                draw_index + 1,
                result.status,
                selected,
                result.finish_reason or result.error,
            )
            GENERATION_LOG.clear()
            gc.collect()
            torch.cuda.empty_cache()

## 7. Summarize and export
Successful generation alone does not establish richness gains; A/P annotation is required.

In [ ]:
from collections import Counter

records = [
    Attempt.model_validate_json(p.read_text(encoding="utf-8"))
    for p in sorted(OUTPUT.glob("Q*__*__r*.json"))
]
print("Records:", len(records), "/ 36")
print("Status:", dict(Counter(r.status for r in records)))
print("Conditions:", dict(Counter(r.condition for r in records)))
from autonomous_modality.answer_diagnostics import diagnose_answer

for record in records:
    diagnostics = diagnose_answer(record.answer or "")
    print("Lexical diagnostics (not A/P scores):", diagnostics.model_dump())
    print(
        record.question_id, record.condition, record.draw_index + 1, record.status, record.selected
    )
print("Development target only. Assess A/P before concluding that the agent improves richness.")

In [ ]:
import shutil

from google.colab import files

archive = shutil.make_archive(
    str(RESULTS_ROOT / f"{RUN_LABEL}_results"),
    "zip",
    root_dir=RESULTS_ROOT,
    base_dir=RUN_LABEL,
)
files.download(archive)